# Road Following 

If you've run through the collision avoidance sample, your should be familiar following three steps

1.  Data collection
2.  Training
3.  Deployment

In this notebook, we'll do the same exact thing!  Except, instead of classification, you'll learn a different fundamental technique, **regression**, that we'll use to
enable JetBot to follow a road (or really, any path or target point).  

1. Place the JetBot in different positions on a path (offset from center, different angles, etc)

>  Remember from collision avoidance, data variation is key!

2. Display the live camera feed from the robot
3. Using a gamepad controller, place a 'green dot', which corresponds to the target direction we want the robot to travel, on the image.
4. Store the X, Y values of this green dot along with the image from the robot's camera

Then, in the training notebook, we'll train a neural network to predict the X, Y values of our label.  In the live demo, we'll use
the predicted X, Y values to compute an approximate steering value (it's not 'exactly' an angle, as
that would require image calibration, but it's roughly proportional to the angle so our controller will work fine).

So how do you decide exactly where to place the target for this example?  Here is a guide we think may help

1.  Look at the live video feed from the camera
2.  Imagine the path that the robot should follow (try to approximate the distance it needs to avoid running off road etc.)
3.  Place the target as far along this path as it can go so that the robot could head straight to the target without 'running off' the road.

> For example, if we're on a very straight road, we could place it at the horizon.  If we're on a sharp turn, it may need to be placed closer to the robot so it doesn't run out of boundaries.

Assuming our deep learning model works as intended, these labeling guidelines should ensure the following:

1.  The robot can safely travel directly towards the target (without going out of bounds etc.)
2.  The target will continuously progress along our imagined path

What we get, is a 'carrot on a stick' that moves along our desired trajectory.  Deep learning decides where to place the carrot, and JetBot just follows it :)

### Labeling example video

Execute the block of code to see an example of how to we labeled the images.  This model worked after only 123 images :)

In [1]:
from IPython.display import HTML
HTML('<iframe width="560" height="315" src="https://www.youtube.com/embed/FW4En6LejhI" frameborder="0" allow="accelerometer; autoplay; encrypted-media; gyroscope; picture-in-picture" allowfullscreen></iframe>')

/home/olek/sem5/myenv/lib/python3.12/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


### Import Libraries

So lets get started by importing all the required libraries for "data collection" purpose. We will mainly use OpenCV to visualize and save image with labels. Libraries such as uuid, datetime are used for image naming. 

In [2]:
import ipywidgets as widgets
from IPython.display import display
from uuid import uuid1
import os
import glob
import numpy as np
import cv2

### Browse & Annotate Dataset Images

Instead of a live camera, we iterate over images already collected in `put_jetbot_dataset`.
For each image:
1. The **left** panel shows the raw image; the **right** panel shows it with the annotation overlay (green arrow).
2. Drag the **forward** slider (up/down speed) and **left (turn)** slider (steering).
3. The arrow on the right image shows the intended motion: pointing up = drive forward, leaning left/right = turn.
4. Click **Save & Next** to write the image + label to `dataset_annotated/` and advance, or **Skip** to discard and advance.

In [6]:
import datetime
from collections import OrderedDict
import time

try:
    from ipyevents import Event
    HAS_IPYEVENTS = True
except ImportError:
    HAS_IPYEVENTS = False
    print("ipyevents not installed — click-to-annotate unavailable. Run: pip install ipyevents")

import ipywidgets as widgets
from IPython.display import display
import os, glob
import numpy as np
import cv2

INPUT_DIR  = 'put_jetbot_dataset/dataset'
OUTPUT_DIR = 'dataset_annotated_final'
os.makedirs(OUTPUT_DIR, exist_ok=True)

image_paths = sorted(glob.glob(os.path.join(INPUT_DIR, '**/*.jpg'), recursive=True))
print(f"Found {len(image_paths)} images in {INPUT_DIR}")
print(f"Output dir: {os.path.abspath(OUTPUT_DIR)}")

session_groups = OrderedDict()
for idx, path in enumerate(image_paths):
    session = os.path.basename(os.path.dirname(path))
    session_groups.setdefault(session, []).append(idx)

PROGRESS_FILE = os.path.join(OUTPUT_DIR, '.progress')

def _load_progress():
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE) as f:
            lines = f.read().strip().split('\n')
            return lines[0], int(lines[1]), int(lines[2])
    return datetime.datetime.now().strftime('%Y%m%d_%H%M%S'), 0, 0

def _save_progress(session, idx, frame_count):
    with open(PROGRESS_FILE, 'w') as f:
        f.write(f'{session}\n{idx}\n{frame_count}\n')

session_name, start_index, start_frame = _load_progress()
session_img_dir = os.path.join(OUTPUT_DIR, session_name)
session_csv     = os.path.join(OUTPUT_DIR, session_name + '.csv')
os.makedirs(session_img_dir, exist_ok=True)

state = {'index': start_index, 'frame': start_frame, 'last_click': 0.0}
print(f"Session: {session_name}")
print(f"Resuming from image {start_index+1}/{len(image_paths)}  (annotated so far: {start_frame})")

ARROW_CX, ARROW_CY = 112, 218
ARROW_SCALE_LEFT    = 70
ARROW_SCALE_FWD     = 90

image_widget   = widgets.Image(format='jpeg', width=224, height=224)
target_widget  = widgets.Image(format='jpeg', width=224, height=224)
forward_slider = widgets.FloatSlider(min=-1.0, max=1.0, step=0.01, description='forward', value=0.0,
                                     style={'description_width': 'initial'})
left_slider    = widgets.FloatSlider(min=-1.0, max=1.0, step=0.01, description='left (turn)', value=0.0,
                                     style={'description_width': 'initial'})
auto_save_cb   = widgets.Checkbox(value=True, description='Auto-save on click', indent=False)
progress_label = widgets.Label(value='')
session_widget = widgets.HTML(value='')
save_btn       = widgets.Button(description='Save & Next', button_style='success')
skip_btn       = widgets.Button(description='Skip',        button_style='warning')

def _to_jpeg(bgr):
    return bytes(cv2.imencode('.jpg', bgr)[1])

def _draw_overlay(frame, forward, left):
    img = frame.copy()
    ex = int(ARROW_CX - left    * ARROW_SCALE_LEFT)
    ey = int(ARROW_CY - forward * ARROW_SCALE_FWD)
    cv2.arrowedLine(img, (ARROW_CX, ARROW_CY), (ex, ey), (0, 255, 0), 3, tipLength=0.3)
    cv2.circle(img, (ARROW_CX, ARROW_CY), 5, (0, 0, 255), -1)
    return img

def _load(index):
    frame = cv2.imread(image_paths[index])
    return cv2.resize(frame, (224, 224))

def _session_html():
    current = state['index']
    rows = []
    for sess, indices in session_groups.items():
        done = sum(1 for i in indices if i < current)
        left = len(indices) - done
        color = '#aaffaa' if left == 0 else '#ffffff'
        rows.append(
            f'<tr style="background:{color}">'
            f'<td style="padding:2px 8px">{sess}</td>'
            f'<td style="padding:2px 8px;text-align:center">{done}/{len(indices)}</td>'
            f'<td style="padding:2px 8px;text-align:center">{left} left</td>'
            f'</tr>'
        )
    return (
        '<table style="font-size:12px;border-collapse:collapse">'
        '<tr><th style="padding:2px 8px">Session</th>'
        '<th style="padding:2px 8px">Done</th>'
        '<th style="padding:2px 8px">Remaining</th></tr>'
        + ''.join(rows) + '</table>'
    )

def _refresh():
    i = state['index']
    if i >= len(image_paths):
        progress_label.value = f"Done!  Annotated {state['frame']} / {len(image_paths)} images."
    else:
        progress_label.value = f"Image {i+1} / {len(image_paths)}  |  annotated: {state['frame']}"
        frame = _load(i)
        image_widget.value  = _to_jpeg(frame)
        target_widget.value = _to_jpeg(_draw_overlay(frame, forward_slider.value, left_slider.value))
    session_widget.value = _session_html()

def _on_slider(change):
    i = state['index']
    if i < len(image_paths):
        target_widget.value = _to_jpeg(_draw_overlay(_load(i), forward_slider.value, left_slider.value))

def _on_save(b):
    i = state['index']
    if i >= len(image_paths):
        return
    frame = _load(i)
    fwd, left = forward_slider.value, left_slider.value
    frame_id = state['frame'] + 1
    cv2.imwrite(os.path.join(session_img_dir, f'{frame_id:04d}.jpg'), frame)
    with open(session_csv, 'a') as f:
        f.write(f'{frame_id},{fwd:.6f},{left:.6f}\n')
    state['frame'] += 1
    state['index'] += 1
    forward_slider.value = 0.0
    left_slider.value    = 0.0
    _save_progress(session_name, state['index'], state['frame'])
    _refresh()

def _on_skip(b):
    state['index'] += 1
    _save_progress(session_name, state['index'], state['frame'])
    _refresh()

forward_slider.observe(_on_slider, names='value')
left_slider.observe(_on_slider,    names='value')
save_btn.on_click(_on_save)
skip_btn.on_click(_on_skip)

if HAS_IPYEVENTS:
    # wait=500 throttles to one event per 500 ms at the JS level
    click_event = Event(source=target_widget, watched_events=['click'], wait=200)

    def _on_click(event):
        # secondary debounce guard on the Python side
        now = time.time()
        if now - state['last_click'] < 0.2:
            return
        state['last_click'] = now

        px = event.get('offsetX', ARROW_CX)
        py = event.get('offsetY', ARROW_CY)
        fwd  = round(max(-1.0, min(1.0, (ARROW_CY - py) / ARROW_SCALE_FWD)),  2)
        left = round(max(-1.0, min(1.0, (ARROW_CX - px) / ARROW_SCALE_LEFT)), 2)
        forward_slider.value = fwd
        left_slider.value    = left
        if auto_save_cb.value:
            _on_save(None)

    click_event.on_dom_event(_on_click)
    print("Click-to-annotate enabled — click on the RIGHT image to annotate and auto-save.")
else:
    auto_save_cb.disabled = True

_refresh()
display(
    widgets.HTML('<b>Click the right image to annotate. Sliders for fine adjustment.</b>'),
    widgets.HBox([image_widget, target_widget]),
    forward_slider, left_slider,
    widgets.HBox([save_btn, skip_btn, auto_save_cb]),
    progress_label,
    session_widget,
)

Found 7585 images in put_jetbot_dataset/dataset
Output dir: /home/olek/sem6/robotics2/jetbot-robotics/dataset_annotated_final
Session: 20260616_000911
Resuming from image 6227/7585  (annotated so far: 6226)
Click-to-annotate enabled — click on the RIGHT image to annotate and auto-save.


HTML(value='<b>Click the right image to annotate. Sliders for fine adjustment.</b>')

FloatSlider(value=0.0, description='forward', max=1.0, min=-1.0, step=0.01, style=SliderStyle(description_widt…

FloatSlider(value=0.0, description='left (turn)', max=1.0, min=-1.0, step=0.01, style=SliderStyle(description_…

Label(value='Image 6227 / 7585  |  annotated: 6226')

HTML(value='<table style="font-size:12px;border-collapse:collapse"><tr><th style="padding:2px 8px">Session</th…

### Export

Once annotation is complete, run the cell below to zip `dataset_annotated/` for use in training.
Pass the unzipped folder to training with `python train.py --dataset path/to/dataset_annotated`.

In [4]:
import datetime

def timestr():
    return str(datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S'))

!zip -r -q road_following_{OUTPUT_DIR}_{timestr()}.zip {OUTPUT_DIR}